# 유튜브 구간 얼굴 탐지 (A 이유노)

유튜브 주소와 구간을 넣으면 그 구간만 받아, 사전학습 yolov11n-face로 얼굴 박스와 track_id를 그린 영상(H.264, 원본 소리 포함)을 만든다.

- 시작 전: **런타임 → 런타임 유형 변경 → T4 GPU**
- 공개된 유명인 영상만 쓰고, 출처는 `sources.csv`에 자동으로 기록된다. 받은 영상과 결과는 팀 내부 평가용으로만 쓰고 다시 배포하지 않는다.
- 2번 셀에서 "Sign in to confirm you're not a bot"이 나오면 유튜브가 Colab을 막은 것이다. 브라우저 확장(예: Get cookies.txt LOCALLY)으로 youtube.com 쿠키를 `cookies.txt`로 내보내 왼쪽 📁에 올리고 `COOKIES = "cookies.txt"`로 바꿔 다시 실행한다. 쿠키 파일은 로그인 정보이므로 공유하지 않는다.

In [ ]:
# 1. 준비: 저장소·패키지·가중치 (여러 번 실행해도 됨)
import os

if not os.path.exists("/content/Triple-Y-Mini"):
    !git clone -q https://github.com/Leeyuno0419/Triple-Y-Mini.git /content/Triple-Y-Mini
%cd /content/Triple-Y-Mini
!pip install -q -r requirements.txt
!pip install -q -U yt-dlp  # 유튜브가 자주 바뀌어 항상 최신으로
if not os.path.exists("weights/best.pt"):
    !wget -q -O weights/best.pt https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt
!nvidia-smi -L

In [ ]:
# 2. 입력: 주소와 구간을 바꾸고 실행
URL = "https://www.youtube.com/watch?v=영상ID"
START = "1:00"   # 분:초 (1시간 넘으면 시:분:초)
END = "1:30"     # 30초 안팎을 권장 (길수록 오래 걸림)
COOKIES = ""     # 봇 차단이 뜰 때만 "cookies.txt"

tag = f"{START}-{END}".replace(":", "")
section = f"section_{tag}.mp4"
cookie_opt = f"--cookies {COOKIES}" if COOKIES else ""

# 720p 이하 영상+소리, 지정 구간만 받기. 출처(ID, 주소, 제목, 구간)를 sources.csv에 추가
!rm -f {section}
!yt-dlp -q --no-warnings {cookie_opt} -f "bv*[height<=720][ext=mp4]+ba[ext=m4a]/b[height<=720]" --merge-output-format mp4 --download-sections "*{START}-{END}" --force-keyframes-at-cuts -o "{section}" --print-to-file "%(id)s,%(webpage_url)s,{START}-{END},%(title)s" sources.csv --no-simulate "{URL}"
assert os.path.exists(section), "다운로드 실패: 위 에러 메시지를 확인 (봇 차단이면 위 안내대로 COOKIES 설정)"
!ls -lh {section}

In [ ]:
# 3. 얼굴 탐지: 프레임마다 detect_faces(track=True), 박스·track_id·점수를 그림
import time

import cv2

from src.detector import detect_faces, reset_tracker

cap = cv2.VideoCapture(section)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
drawn = f"drawn_{tag}.mp4"  # 소리 없는 임시 파일
out = cv2.VideoWriter(drawn, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

reset_tracker()  # 새 영상마다 처음에 한 번
ids, frames, total_boxes, start = set(), 0, 0, time.time()
while True:
    ok, frame = cap.read()
    if not ok:
        break
    boxes = detect_faces(frame, track=True)
    for b in boxes:
        x1, y1, x2, y2 = b["xyxy"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"{b['track_id']} {b['conf']:.2f}" if b["track_id"] is not None else f"{b['conf']:.2f}"
        cv2.putText(frame, label, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        if b["track_id"] is not None:
            ids.add(b["track_id"])
    out.write(frame)
    frames += 1
    total_boxes += len(boxes)
cap.release()
out.release()

sec = time.time() - start
print(f"프레임 {frames}, 등장 인원(고유 track_id) {len(ids)}, 프레임당 평균 얼굴 {total_boxes / max(frames, 1):.1f}, 처리 {frames / sec:.1f} fps")

In [ ]:
# 4. H.264로 변환하고 원본 소리를 합침 (소리가 없는 영상이어도 동작)
result = f"result_{tag}.mp4"
!ffmpeg -y -loglevel error -i {drawn} -i {section} -map 0:v -map 1:a? -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest {result}
!rm -f {drawn}
!ls -lh {result}

In [ ]:
# 5. 미리 보기
from IPython.display import Video

Video(result, embed=True, width=720)

In [ ]:
# 6. 내려받기: 결과 영상과 출처 기록
from google.colab import files

files.download(result)
files.download("sources.csv")